# ObjectLab training

This notebook prepares the leakage-reviewed copy (keeping test duplicates and removing lower-priority train/validation copies), then fits the scratch CNN and MobileNetV4 models. It writes results to Google Drive.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'Pillow', 'scikit-learn', 'torch', 'torchvision', 'timm', 'safetensors', 'onnx', 'onnxruntime', 'matplotlib'], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RAW_DATA_DIR = '/content/drive/MyDrive/Computer vision dataset'
OBJECTLAB_DIR = '/content/drive/MyDrive/ObjectLab'
CLEAN_DATA_DIR = OBJECTLAB_DIR + '/dataset_clean'
OUTPUT_DIR = OBJECTLAB_DIR + '/run'


In [ ]:
from pathlib import Path
Path('/content/prepare_dataset.py').write_text('"""Rebuild the leakage-reviewed split copy from the original folder tree.\n\nThis keeps the source dataset untouched and removes only the reviewed paths\nlisted in results/leakage_removals.csv after checking their source SHA-256.\n"""\nimport argparse, csv, hashlib, json, shutil\nfrom pathlib import Path\n\ndef main():\n    ap=argparse.ArgumentParser(); ap.add_argument(\'--source\',required=True); ap.add_argument(\'--output\',required=True)\n    ap.add_argument(\'--removals\',default=str(Path(__file__).resolve().parents[1]/\'results\'/\'leakage_removals.csv\'))\n    args=ap.parse_args(); source=Path(args.source).resolve(); output=Path(args.output).resolve()\n    if output.exists(): raise SystemExit(f\'Output already exists: {output}\')\n    rows=list(csv.DictReader(open(args.removals,newline=\'\',encoding=\'utf-8-sig\')))\n    for row in rows:\n        p=source/Path(row[\'removed_path\'].replace(\'\\\\\',\'/\'))\n        if not p.is_file(): raise SystemExit(f\'Missing source file: {p}\')\n        digest=hashlib.sha256(p.read_bytes()).hexdigest()\n        if digest!=row[\'sha256\']: raise SystemExit(f\'Source hash changed for {row["removed_path"]}: expected {row["sha256"]}, got {digest}\')\n    shutil.copytree(source,output)\n    for row in rows: (output/Path(row[\'removed_path\'].replace(\'\\\\\',\'/\'))).unlink()\n    shutil.copy2(args.removals,output/\'leakage_removals.csv\')\n    summary=Path(args.removals).with_name(\'leakage_removals.json\')\n    if summary.exists(): shutil.copy2(summary,output/summary.name)\n    print(json.dumps({\'source\':str(source),\'prepared_copy\':str(output),\'removed_files\':len(rows),\'source_modified\':False},indent=2))\n\nif __name__==\'__main__\': main()\n', encoding='utf-8')


In [ ]:
from pathlib import Path
Path('/content/leakage_removals.csv').write_text('removed_path,kept_path,phash_hamming,sha256,decoded_rgb_sha256,reason\ntrain\\bottles\\photo_5426963673156099991_x.jpg,test\\bottles\\15.jpg,0,9e5be35dfd0175cb858121096f3ddce46bdf58fa99308a3121bd0e6dd45a87b0,6f0f5ac53c03a84f03344d0eee1d06c4db173596f64bc237b7493801a847810a,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\bottles\\photo_5426963673156099992_y.jpg,val\\bottles\\16.jpg,0,6cd2a010e532248f4f583bd3af5f3e97a6c5787188689c434e7dfad2cc2600f8,f04b1e9b82939c4590bf3c7124a4edd3d20208a23fcc1093781cc4ce645bbb5f,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\bottles\\photo_5426963673156099995_y.jpg,val\\bottles\\28.jpg,0,fd6c7b8ee93a2686a0b8150b95d8395f9cd6dd905b2c345ae2fbbe662d3a28e7,571f0dd19e252ad5658bc0824c279881f62b8eff8f0da027fb5c0b9d4b6eb925,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\bottles\\photo_5426963673156099997_y.jpg,val\\bottles\\22.jpg,0,74f6710ffae1fe79ba48bc587f881592de3dc9240e06731f03ef99b8041c2dfc,d451557ddf585d5b7acba228b52329d2a9590e9861e1e807a629d7ae5172c316,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\bottles\\photo_5426963673156100004_y.jpg,test\\bottles\\4.jpg,0,2c6e1428f381c6e8db51d4a93f2a4ead9a63b36796926b6167fa293123530ff3,b2a303fb9dbf6da688be6abab1de0aa4d8e96cc62084cb35f4191ef6e401b777,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\bottles\\photo_5426963673156100011_y.jpg,val\\bottles\\1.jpg,0,40ceb89c0f88f81039a953d1df3b23cb3dcca74ac44dfd998cc37e3df7b5f628,0871cebf3727391b2a7ce9c53bdeff804e914967530f308cf4d5ebdc02b21596,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\bottles\\photo_5426963673156100012_x.jpg,test\\bottles\\12.jpg,0,726594100597658f4615281d0a44c7182bb96af82ffcd8b67db25e124cc6eb0e,8d685328f4374fcfe0e8cd790fd19bbcbac01db72c3fb5e9829607a8427bb51d,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\bottles\\photo_5426963673156100013_y.jpg,test\\bottles\\30.jpg,0,6388288f48fb800de36edee82f93ce0c025ecc927111e7614b0d57dc89c2f7c8,044f452651dd61f04667eb7c00011999f2a051183f3fe79a05fe6026cc3f5ea4,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\cups\\photo_5426963673156099934_x.jpg,val\\cups\\89.jpg,0,28d13002fb553a1322bfcf151606f221d1b2fc3ed5f64c01b5a8c449511e463a,71386309484ded73cea626fdac8a341119e7490bc94fd980dd7aadf00eafbb5c,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\cups\\photo_5426963673156099937_y.jpg,val\\cups\\94.jpg,0,777ee9103a6a9368262a8dcc6570d8dae5d7fbc96c15f70e79982767d91afc03,b4e99bba85ee3e9a0bba0a9b35db2251e4790f53b1eb254c6a76bed85fd50c6a,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\cups\\photo_5426963673156099938_y.jpg,val\\cups\\92.jpg,2,55d7b8cac16aa37cbe383fc27b055e3c389d18bc0d874b6b80a3eb81b90de986,c6d975522de89bd76ca3577d607f3dfbeffb4185dae46c0fe2e31811e5f90648,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\cups\\photo_5426963673156099940_x.jpg,val\\cups\\91.jpg,0,f61b90a0440bd9f4a57c77f8ee6608326f9c3c109814786e74bd6d74ee0f4ece,17b2c7f07a11ab5c13feb68b4bb4d8e98533928a44e1e0eed3dffd3f9b5fc12b,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\cups\\photo_5426963673156099942_y.jpg,val\\cups\\90.jpg,0,19b75ce48c855b7d20c45653344e0ccea04a572e216c171802d1fdb2fc504677,50c8263ce9d3cbf96b98912a1c2fa96b3503dd77abeb26c0c92750c50c2ec385,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\pens\\photo_5426963673156100071_y.jpg,test\\pens\\10.jpg,0,38ea00e71a3cc11b691bd4d6761687a0490643cc1a1a0e3bc8be9bd43fe547cd,a2375080b4173eec445a5620a318b468182fb01ab0f90ecc2f3c96e28e722d76,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\pens\\photo_5426963673156100097_x.jpg,val\\pens\\28.jpg,0,a57b9630045fb065b8671c33f56df943f05d397f9b40c3328d352ce9452321f9,ce06a863c1f2bad7887b034ec7ee76cff0d238b87b3a0ec41d95499e620ab041,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\scissors\\31.jpg,test\\scissors\\photo_5426963673156100176_y.jpg,0,0b70f8c2c82f8464b4fccf3272a1816c6872e8d231aeaaa7a6b86cc804fe0930,08b7d120c8c0e8c9b6fa7af4504782982c3e6d6cc00d83eb03ea440e2450fe11,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\scissors\\32.jpg,test\\scissors\\photo_5426963673156100177_y.jpg,0,f6415387ea60ba894579f0bc4f8b876b00d4040f7ee13e3b0a2b03c9690cec4c,ce163365d7e8d6f74bfdcf18819e108b9e32a6ebdb0aef85b040bf2a73061066,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\scissors\\photo_5426963673156100137_x.jpg,test\\scissors\\24.jpg,0,6a9c08f973eedfa6d13ca363fc53f9673f857c0108fdf9e4b05c4a4770af404b,d916d15c11ba8fba9f5bd3106beb6caff2801d291760efbc5c2ac45c2647d2f4,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\scissors\\photo_5426963673156100138_y.jpg,val\\scissors\\23.jpg,0,a40182b4aee0ccbf89ec5d079d30f10b46f65627a9f9dd83cfaaed53e0e6049f,75bc13a4f5a4c8ab25a54a4f65de8ea6328d536ef6be836e868668541684306d,visually reviewed cross-split same underlying photo; preserve test > validation > training\nval\\cups\\96.jpg,test\\cups\\photo_5426963673156099965_y.jpg,0,4438d16db745175da71eed5f27b44f8262310f1cc5619a30eacfbba1922b2575,360112bdd7a003fd9b831e5851ec67bb37b13ba1b9ba43237b7096e6332597b8,visually reviewed cross-split same underlying photo; preserve test > validation > training\nval\\scissors\\photo_5426963673156100174_y.jpg,test\\scissors\\1.jpg,0,380d2d54cff969d3dab3742abf52216db717a0aad58d5357155cf71ab300fac6,d8212c28fe91658acc48fb1d2d02576ce174d2e80cd0dbe27744ed35cab70556,visually reviewed cross-split same underlying photo; preserve test > validation > training\ntrain\\scissors\\photo_5426963673156100143_x(1).jpg,train\\scissors\\photo_5426963673156100143_x.jpg,0,75c6345591b532c3ea42de02b425eef19980dcaa4dab2b1ad458a8d3b6f32222,947855297615cd2008ae5cac1b532777439bbbf08b3920f68a3d7f2bff08d95b,same-split byte-identical redundant file; deterministic shortest unsuffixed path retained\n', encoding='utf-8')


In [ ]:
from pathlib import Path
import csv, subprocess, sys
removed = list(csv.DictReader(open('/content/leakage_removals.csv', encoding='utf-8-sig')))
source_count = sum(1 for p in Path(RAW_DATA_DIR).rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.webp','.bmp'})
removed_sources = [Path(RAW_DATA_DIR) / Path(r['removed_path'].replace('\\','/')) for r in removed]
if Path(CLEAN_DATA_DIR).exists():
    clean_count = sum(1 for p in Path(CLEAN_DATA_DIR).rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.webp','.bmp'})
    if clean_count != 623: raise RuntimeError(f'Existing clean copy has {clean_count} images; expected 623.')
elif source_count == 645 and all(p.is_file() for p in removed_sources):
    subprocess.run([sys.executable, '/content/prepare_dataset.py', '--source', RAW_DATA_DIR, '--output', CLEAN_DATA_DIR, '--removals', '/content/leakage_removals.csv'], check=True)
elif source_count == 623:
    CLEAN_DATA_DIR = RAW_DATA_DIR  # The owner already removed the 22 reviewed duplicates.
else:
    raise RuntimeError(f'Expected the 645-image raw snapshot or its 623-image cleaned copy, found {source_count} images.')


In [ ]:
from pathlib import Path
Path('/content/train.py').write_text('"""Reproduce Assignment 1 CNN and train a MobileNetV4 transfer baseline.\n\nThe input contract is RGB float pixels in [0,1]. Preprocessing mirrors A1:\nEXIF transpose, RGB conversion, Lanczos aspect-preserving resize, grey-127 pad.\n"""\nfrom __future__ import annotations\nimport argparse, csv, hashlib, json, math, os, random, time\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageOps\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\nimport torchvision.transforms.functional as TF\nfrom torchvision.transforms import InterpolationMode\nfrom sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report\n\nCLASSES = [\'bottles\', \'cups\', \'pens\', \'scissors\']\nSPLITS = [\'train\', \'val\', \'test\']\nSIZE = 224\nSEED = 2\n# Stable id for the baseline implementation already in the active run before\n# the cached-feature SOTA optimization was added. Baseline behavior is unchanged.\nBASELINE_CODE_ID = \'c3365deed838b2ba637a1e06b8f10175ca0ece8c2da4ed3cc5e6fc97223eccba\'\n\ndef seed_all(seed=SEED):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n    torch.set_num_threads(min(6, max(1, os.cpu_count() or 1)))\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\ndef image_tensor(path):\n    with Image.open(path) as im:\n        im = ImageOps.exif_transpose(im).convert(\'RGB\')\n        canvas = ImageOps.pad(im, (SIZE,SIZE), method=Image.Resampling.LANCZOS, color=(127,127,127))\n        return TF.to_tensor(canvas)\n\ndef image_identity(path):\n    """Encoding-independent decoded RGB digest and 64-bit DCT pHash."""\n    with Image.open(path) as src:\n        im=ImageOps.exif_transpose(src).convert(\'RGB\')\n    pixel_sha=hashlib.sha256((f\'{im.width}x{im.height}:\'.encode())+im.tobytes()).hexdigest()\n    gray=np.asarray(im.convert(\'L\').resize((32,32),Image.Resampling.LANCZOS),dtype=np.float64)\n    n=32; k=np.arange(n)[:,None]; t=np.arange(n)[None,:]\n    basis=np.cos(np.pi/n*(t+0.5)*k); basis[0,:]*=1/math.sqrt(2); basis*=math.sqrt(2/n)\n    coeff=basis@gray@basis.T; low=coeff[:8,:8].flatten(); median=np.median(low[1:])\n    bits=low>median; phash=f\'{sum(int(bit)<<idx for idx,bit in enumerate(bits)):016x}\'\n    return pixel_sha,phash\n\ndef hamming_hex(a,b): return (int(a,16)^int(b,16)).bit_count()\n\ndef augment_tensor(x,rng=random):\n    if rng.random()<.5: x=TF.hflip(x)\n    angle=rng.uniform(-18,18)\n    x=TF.rotate(x,angle,interpolation=InterpolationMode.BILINEAR,fill=[127/255]*3)\n    factor=rng.uniform(.9,1.1); mean=x.mean((1,2),keepdim=True)\n    return ((x-mean)*factor+mean).clamp(0,1)\n\nclass FolderSet(Dataset):\n    def __init__(self, root, split, augment=False, valid_rel=None):\n        self.items=[]; self.augment=augment\n        root=Path(root)\n        for ci, cls in enumerate(CLASSES):\n            d=Path(root)/split/cls\n            for p in sorted(d.iterdir()):\n                if p.is_file() and p.suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'} and (valid_rel is None or p.relative_to(root) in valid_rel): self.items.append((p,ci))\n    def __len__(self): return len(self.items)\n    def __getitem__(self,i):\n        p,y=self.items[i]; x=image_tensor(p)\n        if self.augment: x=augment_tensor(x)\n        return x,y,str(p)\n\nclass ScratchCNN(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.convs=nn.ModuleList([nn.Conv2d(3,16,3,padding=1),nn.Conv2d(16,32,3,padding=1),nn.Conv2d(32,64,3,padding=1)])\n        self.pool=nn.MaxPool2d(2); self.fc1=nn.Linear(64,64); self.drop=nn.Dropout(.3); self.fc2=nn.Linear(64,4)\n    def features(self,x):\n        for c in self.convs: x=self.pool(torch.relu(c(x)))\n        return x\n    def forward(self,x):\n        f=self.features(x); z=f.mean((2,3)); h=torch.relu(self.fc1(z)); return self.fc2(self.drop(h))\n\nclass TransferNet(nn.Module):\n    def __init__(self, backbone):\n        super().__init__(); self.backbone=backbone; self.channels=backbone.feature_info.channels()[-1]; self.head=nn.Linear(self.channels,4)\n    def train(self, mode=True):\n        super().train(mode); self.backbone.eval(); return self\n    def feature_map(self,x): return self.backbone(x)[0]\n    def forward(self,x): return self.head(self.feature_map(x).mean((2,3)))\n\nclass CachedHeadTrainer(nn.Module):\n    """Shares TransferNet state names while training a frozen backbone\'s cached GAP vectors."""\n    def __init__(self, backbone, head=None):\n        super().__init__(); self.backbone=backbone; self.head=head or nn.Linear(backbone.feature_info.channels()[-1],4)\n        for p in self.backbone.parameters(): p.requires_grad=False\n        self.backbone.eval()\n    def train(self,mode=True): super().train(mode); self.backbone.eval(); return self\n    def forward(self,features): return self.head(features)\n\nclass FeatureSet(Dataset):\n    def __init__(self,features,labels,paths): self.features=features;self.labels=labels;self.paths=paths\n    def __len__(self): return len(self.labels)\n    def __getitem__(self,i):\n        view=random.randrange(self.features.shape[1]) if self.features.shape[1]>1 else 0\n        return self.features[i,view],int(self.labels[i]),self.paths[i]\n\ndef cache_feature_bank(dataset, model, mean, std, device, n_views=1, augment_views=False, batch_size=16, label=\'features\'):\n    model.eval(); inputs=[]; metadata=[]; feats=[]; encoded=0\n    def encode_pending():\n        nonlocal encoded,inputs\n        if not inputs: return\n        batch=torch.stack(inputs).to(device); batch=(batch-mean.to(device))/std.to(device)\n        with torch.no_grad(): pooled=model(batch)[0].mean((2,3)).cpu()\n        feats.extend(pooled.unbind()); encoded+=len(inputs); inputs=[]\n        if (encoded//batch_size)%10==0: print(f\'{label}: encoded {encoded}/{len(dataset.items)*n_views}\',flush=True)\n    for i,(path,y) in enumerate(dataset.items):\n        views=[]; original=image_tensor(path); views.append(original)\n        for vi in range(1,n_views):\n            rng=random.Random(SEED+7919*i+104729*vi)\n            views.append(augment_tensor(original.clone(),rng) if augment_views else original.clone())\n        metadata.append((y,str(path)))\n        for view in views:\n            inputs.append(view)\n            if len(inputs)>=batch_size: encode_pending()\n    encode_pending()\n    n=len(dataset.items); x=torch.stack(feats).reshape(n,n_views,-1)\n    return x,[y for y,_ in metadata],[p for _,p in metadata]\n\ndef augmentation_batch(x):\n    # Applied per-item for compatibility across torchvision versions.\n    return x\n\ndef metrics(logits, ys):\n    pred=np.asarray(logits).argmax(1); y=np.asarray(ys)\n    p,r,f,_=precision_recall_fscore_support(y,pred,labels=range(4),zero_division=0)\n    return {\'accuracy\':float(accuracy_score(y,pred)),\'macro_precision\':float(p.mean()),\'macro_recall\':float(r.mean()),\'macro_f1\':float(f.mean()),\n            \'per_class\':{c:{\'precision\':float(p[i]),\'recall\':float(r[i]),\'f1\':float(f[i])} for i,c in enumerate(CLASSES)},\n            \'confusion_matrix\':confusion_matrix(y,pred,labels=range(4)).tolist()}\n\ndef run_epoch(model, loader, optimizer, device, mean=None, std=None):\n    training=optimizer is not None; model.train(training); loss_sum=0.; sample_count=0; logits_all=[]; ys=[]; rows=[]\n    for x,y,paths in loader:\n        x=x.to(device); y=y.to(device)\n        if mean is not None: x=(x-mean)/std\n        if training: optimizer.zero_grad(set_to_none=True)\n        with torch.set_grad_enabled(training):\n            out=model(x); loss=nn.functional.cross_entropy(out,y)\n            if training: loss.backward(); optimizer.step()\n        n=len(y); loss_sum+=float(loss.detach().cpu())*n; sample_count+=n\n        logits_all.extend(out.detach().cpu().numpy()); ys.extend(y.cpu().numpy()); rows.extend(paths)\n    return loss_sum/sample_count, metrics(logits_all,ys), logits_all, ys, rows\n\ndef fit(model, train_loader, val_loader, device, outdir, name, mean=None,std=None, epochs=30, signature=None):\n    model.to(device); optimizer=torch.optim.Adam((p for p in model.parameters() if p.requires_grad),lr=1e-3)\n    best=float(\'inf\'); best_state=None; wait=0; plateau=0; lr_wait=0; lr=1e-3; history=[]\n    for epoch in range(1,epochs+1):\n        t=time.time(); tl,tm,*_=run_epoch(model,train_loader,optimizer,device,mean,std)\n        vl,vm,*_=run_epoch(model,val_loader,None,device,mean,std)\n        history.append({\'epoch\':epoch,\'train_loss\':tl,\'train_accuracy\':tm[\'accuracy\'],\'val_loss\':vl,\'val_accuracy\':vm[\'accuracy\'],\'lr\':optimizer.param_groups[0][\'lr\'],\'seconds\':time.time()-t})\n        print(f\'{name} epoch {epoch}: train loss {tl:.4f} acc {tm["accuracy"]:.3f}; val loss {vl:.4f} acc {vm["accuracy"]:.3f}\',flush=True)\n        if vl < best-1e-12:\n            best=vl; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0; lr_wait=0\n        else:\n            wait+=1; lr_wait+=1\n            if lr_wait>=3:\n                lr=max(optimizer.param_groups[0][\'lr\']*.5,1e-6)\n                for pg in optimizer.param_groups: pg[\'lr\']=lr\n                lr_wait=0\n            if wait>=7: break\n    model.load_state_dict(best_state); outdir.mkdir(parents=True,exist_ok=True)\n    torch.save(model.state_dict(),outdir/f\'{name}.pt\')\n    (outdir/f\'{name}.meta.json\').write_text(json.dumps({\'signature\':signature},indent=2))\n    with (outdir.parent/\'results\'/f\'{name}_history.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=history[0].keys());w.writeheader();w.writerows(history)\n    try:\n        import matplotlib.pyplot as plt\n        fig,axes=plt.subplots(1,2,figsize=(9,3.5))\n        for ax,field,title in zip(axes,[\'loss\',\'accuracy\'],[\'Loss\',\'Accuracy\']):\n            ax.plot([r[\'epoch\'] for r in history],[r[f\'train_{field}\'] for r in history],label=\'Train\')\n            ax.plot([r[\'epoch\'] for r in history],[r[f\'val_{field}\'] for r in history],label=\'Validation\')\n            ax.set(xlabel=\'Epoch\',ylabel=title,title=title);ax.legend()\n        fig.tight_layout();fig.savefig(outdir.parent/\'results\'/f\'{name}_curves.png\',dpi=150);plt.close(fig)\n    except ImportError: pass\n    return model,history\n\nclass BaselineExport(nn.Module):\n    """Adds analytical Grad-CAM: exact gradient chain through GAP->Dense64 ReLU."""\n    def __init__(self, model): super().__init__(); self.model=model\n    def forward(self,x):\n        f=self.model.features(x); z=f.mean((2,3)); h=torch.relu(self.model.fc1(z)); logits=self.model.fc2(h); probs=torch.softmax(logits,1)\n        # d logit_c / d GAP_j, including Dense64 ReLU gate.\n        weights=torch.matmul(self.model.fc2.weight.unsqueeze(0), self.model.fc1.weight.unsqueeze(0) * (h>0).to(h.dtype).unsqueeze(2))\n        weights=weights/(f.shape[2]*f.shape[3])\n        # weights [N,4,64], CAM=sum_j weights[c,j]*feature[j,u,v]\n        cam=torch.einsum(\'ncj,njhw->nchw\',weights,f).relu()\n        flat=cam.flatten(2); lo=flat.amin(2).view(-1,4,1,1); hi=flat.amax(2).view(-1,4,1,1)\n        cam=(cam-lo)/(hi-lo+1e-8)\n        return probs,cam\n\nclass TransferExport(nn.Module):\n    def __init__(self, model, mean, std): super().__init__(); self.model=model; self.register_buffer(\'mean\',mean); self.register_buffer(\'std\',std)\n    def forward(self,x):\n        xn=(x-self.mean)/self.std\n        # For GAP + linear classifier CAM is exactly Grad-CAM before ReLU.\n        fmap=self.model.feature_map(xn)\n        pooled=fmap.mean((2,3)); logits=self.model.head(pooled); probs=torch.softmax(logits,1)\n        cam=(torch.einsum(\'cj,njhw->nchw\',self.model.head.weight,fmap)/(fmap.shape[2]*fmap.shape[3])).relu()\n        flat=cam.flatten(2); lo=flat.amin(2).view(-1,4,1,1); hi=flat.amax(2).view(-1,4,1,1)\n        return probs,(cam-lo)/(hi-lo+1e-8)\n\ndef export_validate(wrapper, model, path, sample, device):\n    import onnx, onnxruntime as ort\n    wrapper.eval().cpu(); sample=sample.cpu()\n    torch.onnx.export(wrapper,sample,str(path),input_names=[\'images\'],output_names=[\'probabilities\',\'heatmaps\'],opset_version=17,dynamic_axes={\'images\':{0:\'batch\'},\'probabilities\':{0:\'batch\'},\'heatmaps\':{0:\'batch\'}},dynamo=False)\n    proto=onnx.load(str(path))\n    if proto.ir_version>10: proto.ir_version=10\n    for op_set in proto.opset_import:\n        if op_set.domain in (\'\',\'ai.onnx\') and op_set.version>17: raise RuntimeError(f\'ONNX opset {op_set.version} exceeds Android target 17\')\n    props={\'classes\':json.dumps(CLASSES),\'input_contract\':\'float32 RGB [0,1], NCHW 224x224\',\'probability_contract\':\'softmax probabilities ordered as classes\',\'heatmap_contract\':\'per-class positive Grad-CAM, min-max normalized to [0,1]\',\'architecture\':(\'scratch_cnn_16_32_64\' if isinstance(wrapper,BaselineExport) else \'mobilenetv4_conv_small.e2400_r224_in1k_gap_linear\')}\n    for key,value in props.items():\n        entry=proto.metadata_props.add(); entry.key=key; entry.value=value\n    onnx.save(proto,str(path)); onnx.checker.check_model(proto)\n    sess=ort.InferenceSession(str(path),providers=[\'CPUExecutionProvider\'])\n    parity={}\n    for batch_n in [1,2]:\n        xb=sample.repeat(batch_n,1,1,1)\n        actual=sess.run(None,{\'images\':xb.numpy()})\n        with torch.no_grad(): expected=wrapper(xb)\n        diffs=[float(np.max(np.abs(a-b.numpy()))) for a,b in zip(actual,expected)]\n        parity[str(batch_n)]=diffs\n        if max(diffs)>2e-4: raise RuntimeError(f\'ONNX parity failed for N={batch_n}: {diffs}\')\n    # Independently verify analytical CAM maps against autograd Grad-CAM.\n    with torch.enable_grad():\n        if isinstance(wrapper,BaselineExport):\n            fmap=model.features(sample).detach().requires_grad_(True)\n            pooled=fmap.mean((2,3)); hidden=torch.relu(model.fc1(pooled)); logits=model.fc2(hidden)\n        else:\n            norm=(sample-wrapper.mean)/wrapper.std\n            fmap=model.feature_map(norm).detach().requires_grad_(True)\n            pooled=fmap.mean((2,3)); logits=model.head(pooled)\n        maps=[]\n        for ci in range(4):\n            grad=torch.autograd.grad(logits[0,ci],fmap,retain_graph=True)[0]\n            cam=(grad.mean((2,3),keepdim=True)*fmap).sum(1,keepdim=True).relu()\n            cam=(cam-cam.amin((2,3),keepdim=True))/(cam.amax((2,3),keepdim=True)-cam.amin((2,3),keepdim=True)+1e-8)\n            maps.append(cam)\n        gradcam=torch.cat(maps,dim=1)\n    camdiff=float(np.max(np.abs(expected[1].numpy()-gradcam.detach().numpy())))\n    if camdiff>2e-4: raise RuntimeError(f\'Analytical Grad-CAM mismatch {path.name}: {camdiff}\')\n    print(f\'ONNX parity {path.name}: N1/N2 max abs {parity}; autograd Grad-CAM max abs {camdiff}\',flush=True)\n\ndef main():\n    ap=argparse.ArgumentParser(); ap.add_argument(\'--data\',required=True); ap.add_argument(\'--out\',required=True); ap.add_argument(\'--epochs\',type=int,default=30); ap.add_argument(\'--batch-size\',type=int,default=16); ap.add_argument(\'--pretrained-weights\',default=os.environ.get(\'OBJECTLAB_PRETRAINED_WEIGHTS\')); ap.add_argument(\'--audit-only\',action=\'store_true\',help=\'Write the image manifest, decoded hashes and cross-split duplicate candidates, then stop before training.\'); ap.add_argument(\'--reviewed-candidates\',help=\'CSV reviewed by a human with the same candidate pairs and a nonempty decision column.\')\n    args=ap.parse_args(); seed_all(); root=Path(args.data); out=Path(args.out); ml=out/\'ml\'; models=out/\'models\'; results=out/\'results\'\n    for p in [ml,models,results]:p.mkdir(parents=True,exist_ok=True)\n    mpl_cache=Path.cwd()/\'work\'/\'matplotlib_cache\'; mpl_cache.mkdir(parents=True,exist_ok=True); os.environ[\'MPLCONFIGDIR\']=str(mpl_cache)\n    audit=[]; seen={}; pixel_seen={}; bad=[]; cross_split_dups=[]; within_split_duplicates=[]; duplicate_hashes=[]\n    for split in SPLITS:\n        for ci,c in enumerate(CLASSES):\n            files=sorted(p for p in (root/split/c).iterdir() if p.is_file() and p.suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'})\n            for p in files:\n                try:\n                    with Image.open(p) as im: im.verify()\n                except Exception as e:\n                    bad.append({\'path\':str(p),\'error\':str(e)}); continue\n                digest=hashlib.sha256(p.read_bytes()).hexdigest()\n                pixel_digest,phash=image_identity(p)\n                audit.append({\'path\':str(p.relative_to(root)),\'split\':split,\'class\':c,\'sha256\':digest,\'decoded_rgb_sha256\':pixel_digest,\'phash64\':phash})\n                if digest in seen:\n                    prior=seen[digest]; duplicate_hashes.append(digest)\n                    if prior[0]!=split: cross_split_dups.append({\'sha256\':digest,\'first\':prior[2],\'second\':str(p.relative_to(root))})\n                    else: within_split_duplicates.append({\'sha256\':digest,\'split\':split,\'class_first\':prior[1],\'class_second\':c,\'first\':prior[2],\'second\':str(p.relative_to(root))})\n                    print(\'DUPLICATE_HASH\',prior[2],p,flush=True)\n                else: seen[digest]=(split,c,str(p.relative_to(root)))\n                if pixel_digest in pixel_seen and pixel_seen[pixel_digest][0]!=split:\n                    prior=pixel_seen[pixel_digest]; cross_split_dups.append({\'kind\':\'decoded_pixels\',\'first\':prior[2],\'second\':str(p.relative_to(root))})\n                elif pixel_digest not in pixel_seen: pixel_seen[pixel_digest]=(split,c,str(p.relative_to(root)))\n    with (results/\'dataset_manifest.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=audit[0].keys());w.writeheader();w.writerows(audit)\n    counts={s:{c:sum(r[\'split\']==s and r[\'class\']==c for r in audit) for c in CLASSES} for s in SPLITS}\n    near=[]\n    for i,a in enumerate(audit):\n        for b in audit[i+1:]:\n            if a[\'split\']!=b[\'split\']:\n                dist=hamming_hex(a[\'phash64\'],b[\'phash64\'])\n                if dist<=6: near.append({\'first\':a[\'path\'],\'first_split\':a[\'split\'],\'first_class\':a[\'class\'],\'second\':b[\'path\'],\'second_split\':b[\'split\'],\'second_class\':b[\'class\'],\'phash_hamming\':dist})\n    with (results/\'near_duplicate_candidates.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=[\'first\',\'first_split\',\'first_class\',\'second\',\'second_split\',\'second_class\',\'phash_hamming\']);w.writeheader();w.writerows(near)\n    (results/\'dataset_audit.json\').write_text(json.dumps({\'class_order\':CLASSES,\'counts\':counts,\'image_count\':len(audit),\'invalid_images\':bad,\'exact_duplicate_hash_occurrences\':len(duplicate_hashes),\'within_split_exact_duplicate_pairs\':within_split_duplicates,\'cross_split_exact_duplicate_pairs\':cross_split_dups,\'cross_split_phash_candidates_le_6_bits\':len(near)},indent=2))\n    fingerprint=hashlib.sha256(\'\'.join(f"{r[\'path\']}:{r[\'sha256\']}\\n" for r in audit).encode()).hexdigest()\n    fpfile=results/\'dataset_fingerprint.txt\'\n    if fpfile.exists() and fpfile.read_text().strip()!=fingerprint:\n        if not args.audit_only: raise RuntimeError(\'Dataset content/split fingerprint changed. Run --audit-only on the new dataset, then review audit gates before training.\')\n    fpfile.write_text(fingerprint+\'\\n\')\n    if args.audit_only:\n        print(\'Dataset audit complete; training was not started.\',flush=True)\n        return\n    if cross_split_dups:\n        raise RuntimeError(f\'Found {len(cross_split_dups)} cross-split exact file/decoded-pixel duplicates. Resolve split leakage before training; see results/dataset_audit.json.\')\n    if near:\n        if not args.reviewed_candidates:\n            raise RuntimeError(f\'Found {len(near)} cross-split pHash candidates. Review results/near_duplicate_candidates.csv, save a reviewed copy with a nonempty decision column, then pass --reviewed-candidates PATH.\')\n        with Path(args.reviewed_candidates).open(newline=\'\',encoding=\'utf-8-sig\') as f:\n            reviewed=list(csv.DictReader(f))\n        req={\'first\',\'second\',\'phash_hamming\',\'decision\'}\n        if not reviewed or not req.issubset(reviewed[0]): raise RuntimeError(\'Reviewed candidate CSV must include first, second, phash_hamming, and decision columns.\')\n        expected={(r[\'first\'],r[\'second\'],str(r[\'phash_hamming\'])) for r in near}\n        actual={(r[\'first\'],r[\'second\'],str(int(r[\'phash_hamming\']))) for r in reviewed if r.get(\'decision\',\'\').strip()}\n        if expected!=actual: raise RuntimeError(\'Reviewed candidate pairs do not exactly match current audit or some decisions are blank; review the full current candidate list.\')\n    valid_rel={Path(r[\'path\']) for r in audit}\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\'); print(\'device\',device,\'counts\',counts,flush=True)\n    code_hash=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    def signature(name,augmentation,pretrained_hash=None):\n        code_id=BASELINE_CODE_ID if name==\'baseline_aug_1\' else code_hash\n        payload={\'dataset_fingerprint\':fingerprint,\'training_code_sha256\':code_id,\'name\':name,\'augmentation\':augmentation,\'seed\':SEED,\'max_epochs\':args.epochs,\'batch_size\':args.batch_size,\'optimizer\':\'Adam\',\'learning_rate\':1e-3,\'early_stopping_patience\':7,\'lr_plateau_patience\':3,\'pretrained_sha256\':pretrained_hash}\n        return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()\n    def load_checkpoint(model,name,expected_signature):\n        pt=models/f\'{name}.pt\'; meta=models/f\'{name}.meta.json\'\n        if not pt.exists(): return False\n        if not meta.exists() or json.loads(meta.read_text()).get(\'signature\')!=expected_signature:\n            raise RuntimeError(f\'Checkpoint {pt.name} has no matching dataset/code/config signature; move it aside before training.\')\n        model.load_state_dict(torch.load(pt,map_location=\'cpu\',weights_only=True)); return True\n    test=FolderSet(root,\'test\',valid_rel=valid_rel); val=FolderSet(root,\'val\',valid_rel=valid_rel)\n    tl=DataLoader(test,batch_size=args.batch_size,shuffle=False,num_workers=0); vl=DataLoader(val,batch_size=args.batch_size,shuffle=False,num_workers=0)\n    # A1 architecture reproduction; checkpoint reuse makes export retries inexpensive.\n    base_sig=signature(\'baseline_aug_1\',True)\n    seed_all(); baseline=ScratchCNN()\n    if load_checkpoint(baseline,\'baseline_aug_1\',base_sig): baseline.to(device)\n    else:\n        tr=DataLoader(FolderSet(root,\'train\',True,valid_rel),batch_size=args.batch_size,shuffle=True,num_workers=0)\n        baseline,_=fit(baseline,tr,vl,device,models,\'baseline_aug_1\',epochs=args.epochs,signature=base_sig)\n    baseline.to(device)\n    _,m,lg,ys,paths=run_epoch(baseline,tl,None,device); save_eval(results,\'baseline_aug_1\',m,lg,ys,paths,root)\n    # ImageNet-pretrained MobileNetV4 Conv Small, frozen visual features + 4-way GAP linear head.\n    cache=Path.cwd()/\'work\'/\'model-cache\'; cache.mkdir(parents=True,exist_ok=True)\n    os.environ[\'HF_HOME\']=str(cache/\'huggingface\'); os.environ[\'HUGGINGFACE_HUB_CACHE\']=str(cache/\'huggingface\'/\'hub\'); os.environ[\'TORCH_HOME\']=str(cache/\'torch\')\n    import timm\n    import timm.data\n    torch.manual_seed(SEED)\n    backbone=timm.create_model(\'mobilenetv4_conv_small.e2400_r224_in1k\',pretrained=False,features_only=True,out_indices=(-1,))\n    if args.pretrained_weights and Path(args.pretrained_weights).is_file():\n        from safetensors.torch import load_file\n        state=load_file(args.pretrained_weights,device=\'cpu\')\n        incompatible=backbone.load_state_dict(state,strict=False)\n        if incompatible.missing_keys: raise RuntimeError(f\'Missing pretrained feature weights: {incompatible.missing_keys}\')\n        pretrained_sha256=hashlib.sha256(Path(args.pretrained_weights).read_bytes()).hexdigest()\n        print(f"Loaded {len(state)-len(incompatible.unexpected_keys)} MobileNetV4 feature tensors from {args.pretrained_weights}; discarded classifier/head tensors {incompatible.unexpected_keys}",flush=True)\n    else:\n        # Normal CLI route: timm resolves and downloads the published ImageNet checkpoint.\n        backbone=timm.create_model(\'mobilenetv4_conv_small.e2400_r224_in1k\',pretrained=True,features_only=True,out_indices=(-1,))\n        pretrained_sha256=None\n    if pretrained_sha256 is None:\n        h=hashlib.sha256()\n        for key,tensor in sorted(backbone.state_dict().items()): h.update(key.encode()); h.update(tensor.detach().cpu().contiguous().numpy().tobytes())\n        pretrained_sha256=h.hexdigest()\n    for p in backbone.parameters():p.requires_grad=False\n    backbone.eval(); cfg=timm.data.resolve_model_data_config(backbone); mean=torch.tensor(cfg[\'mean\']).view(1,3,1,1); std=torch.tensor(cfg[\'std\']).view(1,3,1,1)\n    # Frozen feature caching keeps the MobileNet experiment practical on CPU.\n    # The ON arm precomputes four seeded augmented views plus original per image;\n    # each epoch samples exactly one of the five views for each training example.\n    train_source=FolderSet(root,\'train\',valid_rel=valid_rel)\n    train_aug_x,train_aug_y,train_paths=cache_feature_bank(train_source,backbone,mean,std,device,n_views=5,augment_views=True,batch_size=args.batch_size,label=\'MobileNetV4 train views\')\n    train_raw_x,train_raw_y,train_raw_paths=cache_feature_bank(train_source,backbone,mean,std,device,n_views=1,batch_size=args.batch_size,label=\'MobileNetV4 train originals\')\n    val_x,val_y,val_paths=cache_feature_bank(val,backbone,mean,std,device,batch_size=args.batch_size,label=\'MobileNetV4 validation\')\n    test_x,test_y,test_paths=cache_feature_bank(test,backbone,mean,std,device,batch_size=args.batch_size,label=\'MobileNetV4 test\')\n    vloader=DataLoader(FeatureSet(val_x,val_y,val_paths),batch_size=args.batch_size,shuffle=False,num_workers=0)\n    sota=None\n    for aug in [True,False]:\n        seed_all(); name=f\'mobilenetv4_aug_{int(aug)}\'; model=TransferNet(backbone); sig=signature(name,aug,pretrained_sha256)\n        if load_checkpoint(model,name,sig): model.to(device)\n        else:\n            fx,fy,fp=(train_aug_x,train_aug_y,train_paths) if aug else (train_raw_x,train_raw_y,train_raw_paths)\n            train_loader=DataLoader(FeatureSet(fx,fy,fp),batch_size=args.batch_size,shuffle=True,num_workers=0)\n            cached_model=CachedHeadTrainer(backbone,model.head)\n            cached_model,_=fit(cached_model,train_loader,vloader,device,models,name,epochs=args.epochs,signature=sig)\n            model.head=cached_model.head; model.to(device)\n        cached_model=CachedHeadTrainer(model.backbone,model.head).to(device)\n        tloader=DataLoader(FeatureSet(test_x,test_y,test_paths),batch_size=args.batch_size,shuffle=False,num_workers=0)\n        _,m,lg,ys,paths=run_epoch(cached_model,tloader,None,device); save_eval(results,name,m,lg,ys,paths,root)\n        if aug: sota=model\n    x=image_tensor(val.items[0][0]).unsqueeze(0)\n    export_validate(BaselineExport(baseline),baseline,models/\'baseline.onnx\',x,device)\n    export_validate(TransferExport(sota,mean,std),sota,models/\'sota.onnx\',x,device)\n    with (results/\'artifact_manifest.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.writer(f); w.writerow([\'path\',\'sha256\',\'bytes\'])\n        for artifact in sorted(models.glob(\'*\')):\n            if artifact.is_file(): w.writerow([str(artifact.relative_to(out)),hashlib.sha256(artifact.read_bytes()).hexdigest(),artifact.stat().st_size])\n    import importlib.metadata as im\n    packages={}\n    for p in [\'torch\',\'torchvision\',\'timm\',\'numpy\',\'Pillow\',\'scikit-learn\',\'onnx\',\'onnxruntime\']:\n        try: packages[p]=im.version(p)\n        except im.PackageNotFoundError: packages[p]=None\n    (results/\'run_config.json\').write_text(json.dumps({\'seed\':SEED,\'max_epochs\':args.epochs,\'batch_size\':args.batch_size,\'optimizer\':\'Adam\',\'learning_rate\':1e-3,\'early_stopping\':{\'monitor\':\'val_loss\',\'patience\':7,\'restore_best_weights\':True},\'lr_plateau\':{\'monitor\':\'val_loss\',\'patience\':3,\'factor\':0.5,\'minimum\':1e-6},\'augmentation\':{\'horizontal_flip_probability\':0.5,\'rotation_degrees\':[-18,18],\'contrast_factor\':[0.9,1.1]},\'ablation\':\'MobileNetV4 train augmentation enabled vs disabled; same seed and schedule\',\'augmentation_bank\':\'For each training image cache original plus four deterministic seeded augmented views; each epoch selects exactly one view per image. Frozen GAP features are precomputed once. Validation/test features use original images only.\',\'backbone\':\'mobilenetv4_conv_small.e2400_r224_in1k\',\'backbone_pretrained\':True,\'backbone_pretrained_sha256\':pretrained_sha256,\'backbone_frozen\':True,\'transfer_head\':\'global average pooling + linear 4 class logits\',\'normalization_mean\':mean.flatten().tolist(),\'normalization_std\':std.flatten().tolist(),\'class_order\':CLASSES,\'device\':str(device),\'packages\':packages},indent=2))\n    print(\'Training and exports complete\',flush=True)\n\ndef save_eval(results,name,m,logits,ys,paths,root):\n    (results/f\'{name}_metrics.json\').write_text(json.dumps(m,indent=2))\n    pred=np.asarray(logits).argmax(1)\n    with (results/f\'{name}_predictions.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.writer(f);w.writerow([\'path\',\'true_label\',\'predicted_label\',\'probabilities\'])\n        for path,y,p,l in zip(paths,ys,pred,logits):w.writerow([str(Path(path).relative_to(root)),CLASSES[y],CLASSES[p],json.dumps(torch.softmax(torch.tensor(l),0).tolist())])\n\nif __name__==\'__main__\': main()\n', encoding='utf-8')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '/content/train.py', '--data', CLEAN_DATA_DIR, '--out', OUTPUT_DIR, '--epochs', '30', '--batch-size', '16'], check=True)
